# Pemeriksaan hubungan linear fitur gambar dan jumlah sel

Notebook ini memakai `dataset_fitur_jumlah_sel.csv` (fitur `fitur_...` sebagai input dan `Jumlah Sel (sel/mL)` sebagai target). Dataset yang diperiksa memiliki 101 gambar dan 169.000 fitur per gambar. Karena jumlah fitur jauh lebih besar daripada jumlah sampel, korelasi satu fitur dan evaluasi model di bawah adalah diagnostik awal, bukan bukti bahwa seluruh hubungan bersifat linear.

**Penting:** pastikan dahulu setiap jumlah sel benar-benar sesuai dengan waktu pengambilan gambar. Nama file gambar bukan bukti bahwa label sensor diukur pada waktu yang sama. CSV ini tidak menyertakan timestamp sensor asal, sehingga kesesuaian tersebut tidak bisa diverifikasi dari file ini saja. Jangan menarik kesimpulan ilmiah sebelum pencocokan label-gambar diperiksa.

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor

file_data = Path(r"D:\00 Kuliah\TA\Experiment\Model\5 ML\dataset_fitur_jumlah_sel.csv")
kolom_nama = "nama_file"
kolom_target = "Jumlah Sel (sel/mL)"

if not file_data.is_file():
    raise FileNotFoundError(f"File dataset tidak ditemukan: {file_data}")

# Baca header dahulu agar fitur disimpan sebagai float32 dan target sebagai float64.
kolom = pd.read_csv(file_data, encoding="utf-8-sig", nrows=0).columns.tolist()
kolom_fitur = [nama for nama in kolom if nama.startswith("fitur_")]
kolom_wajib = {kolom_nama, kolom_target}
if not kolom_wajib.issubset(kolom):
    raise KeyError(f"Kolom wajib tidak ditemukan: {sorted(kolom_wajib - set(kolom))}")
if not kolom_fitur:
    raise ValueError("Tidak ditemukan kolom fitur dengan awalan 'fitur_'.")

dtype_kolom = {nama: np.float32 for nama in kolom_fitur}
dtype_kolom[kolom_nama] = "string"
dtype_kolom[kolom_target] = "string"
df = pd.read_csv(file_data, encoding="utf-8-sig", dtype=dtype_kolom)

# Kolom target kadang diformat dengan titik ribuan (contoh 121.847.700).
target_teks = df[kolom_target].astype("string").str.strip()
format_ribuan = target_teks.str.fullmatch(r"\d{1,3}(?:\.\d{3})+").fillna(False)
target_teks.loc[format_ribuan] = target_teks.loc[format_ribuan].str.replace(".", "", regex=False)
y = pd.to_numeric(target_teks, errors="coerce").to_numpy(dtype=np.float64)
X = df[kolom_fitur].to_numpy(dtype=np.float32, copy=True)

if df[kolom_nama].isna().any() or df[kolom_nama].duplicated().any():
    raise ValueError("Nama gambar kosong atau duplikat; periksa dataset gabungan.")
if not np.isfinite(y).all():
    raise ValueError("Target jumlah sel berisi nilai kosong atau bukan angka.")
if not np.isfinite(X).all():
    raise ValueError("Fitur berisi nilai kosong atau tak hingga.")
if np.unique(y).size < 2:
    raise ValueError("Target tidak memiliki variasi yang cukup untuk analisis.")

# Urutkan berdasarkan timestamp pada nama gambar untuk evaluasi maju dalam waktu.
timestamp_teks = df[kolom_nama].str.extract(
    r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})", expand=False
)
timestamp_gambar = pd.to_datetime(
    timestamp_teks, format="%Y-%m-%d_%H-%M-%S", errors="coerce"
)
if timestamp_gambar.isna().any():
    raise ValueError("Sebagian nama_file tidak memiliki timestamp yang dikenali.")
urutan = np.argsort(timestamp_gambar.to_numpy(), kind="stable")
X = X[urutan]
y = y[urutan]
df = df.iloc[urutan].reset_index(drop=True)

print(f"Jumlah sampel: {len(df)}")
print(f"Jumlah fitur: {len(kolom_fitur)}")
print(f"Rasio sampel terhadap fitur: {len(df) / len(kolom_fitur):.6f}")
print(f"Rentang jumlah sel: {y.min():,.0f} hingga {y.max():,.0f} sel/mL")
print("PERINGATAN: verifikasi pasangan label sensor dengan timestamp gambar sebelum menafsirkan hasil.")

## 1. Korelasi dan pola fitur individual

Pearson mengukur asosiasi linear setiap fitur dengan target secara terpisah. Nilai korelasi tinggi tidak membuktikan bahwa keseluruhan hubungan multivariat linear. Dengan 169.000 fitur dan hanya 101 sampel, banyak korelasi ekstrem dapat muncul secara kebetulan; gunakan bagian ini untuk eksplorasi, bukan memilih fitur berdasarkan seluruh data sebelum validasi.

In [ ]:
# Hitung korelasi Pearson per fitur dalam potongan agar kebutuhan memori terkendali.
korelasi = np.full(X.shape[1], np.nan, dtype=np.float64)
y_center = y - y.mean()
y_ss = np.dot(y_center, y_center)
ukuran_potongan = 4096

for awal in range(0, X.shape[1], ukuran_potongan):
    akhir = min(awal + ukuran_potongan, X.shape[1])
    potongan = X[:, awal:akhir].astype(np.float64)
    potongan -= potongan.mean(axis=0)
    penyebut = np.sqrt(np.sum(potongan * potongan, axis=0) * y_ss)
    pembilang = potongan.T @ y_center
    korelasi[awal:akhir] = np.divide(
        pembilang,
        penyebut,
        out=np.full(akhir - awal, np.nan),
        where=penyebut > 0,
    )

tabel_korelasi = pd.DataFrame({"fitur": kolom_fitur, "pearson_r": korelasi})
tabel_korelasi["abs_r"] = tabel_korelasi["pearson_r"].abs()
tabel_korelasi = tabel_korelasi.sort_values("abs_r", ascending=False, na_position="last")
display(tabel_korelasi.head(20).drop(columns="abs_r"))

valid_r = tabel_korelasi["pearson_r"].dropna()
print(f"Fitur konstan/tidak terukur: {tabel_korelasi['pearson_r'].isna().sum()}")
print(f"Median |Pearson r|: {valid_r.abs().median():.4f}")

In [ ]:
# Plot hubungan dan residual untuk beberapa fitur dengan |r| terbesar.
fitur_teratas = tabel_korelasi.dropna(subset=["pearson_r"]).head(3)["fitur"].tolist()
if not fitur_teratas:
    raise ValueError("Tidak ada fitur bervariasi yang dapat diplot.")
fig, axes = plt.subplots(len(fitur_teratas), 2, figsize=(11, 4 * len(fitur_teratas)))
if len(fitur_teratas) == 1:
    axes = np.array([axes])

for baris, nama_fitur in enumerate(fitur_teratas):
    nilai_x = df[nama_fitur].to_numpy(dtype=np.float64)
    koef = np.polyfit(nilai_x, y, deg=1)
    prediksi = np.polyval(koef, nilai_x)
    residual = y - prediksi
    urut_x = np.argsort(nilai_x)

    axes[baris, 0].scatter(nilai_x, y, alpha=0.75)
    axes[baris, 0].plot(nilai_x[urut_x], prediksi[urut_x], color="tab:red")
    axes[baris, 0].set(title=f"{nama_fitur} vs jumlah sel", xlabel=nama_fitur, ylabel=kolom_target)

    axes[baris, 1].scatter(prediksi, residual, alpha=0.75)
    axes[baris, 1].axhline(0, color="black", linestyle="--")
    axes[baris, 1].set(title="Residual garis linear", xlabel="Prediksi", ylabel="Residual")

fig.tight_layout()
plt.show()

## 2. Bandingkan model linear dan nonlinear

Korelasi fitur satu per satu tidak menguji seluruh hubungan. Bagian ini membandingkan Ridge (linear dengan regularisasi) dan SVR RBF (nonlinear), memakai seleksi 20 fitur dan standardisasi **di dalam pipeline** sehingga pemilihan fitur tidak melihat data uji. `TimeSeriesSplit` melatih pada waktu lebih awal dan menguji pada waktu berikutnya; ini lebih sesuai untuk deret waktu daripada pengacakan baris.

Skor hanya diagnostik awal: 101 sampel sangat sedikit, parameter model belum dituning, dan pasangan waktu label-gambar tetap harus diverifikasi. Selisih skor CV bukan uji formal linearitas dan tidak otomatis menentukan algoritma terbaik.

In [ ]:
if len(y) < 6:
    raise ValueError("Diperlukan setidaknya 6 sampel untuk TimeSeriesSplit 5-fold.")

jumlah_fold = 5
cv = TimeSeriesSplit(n_splits=jumlah_fold)
jumlah_fitur_pilih = min(20, X.shape[1])

def pipeline_dasar(estimator):
    return Pipeline(
        [
            ("pilih_fitur", SelectKBest(score_func=f_regression, k=jumlah_fitur_pilih)),
            ("skala_fitur", StandardScaler()),
            ("model", estimator),
        ]
    )

model_linear = pipeline_dasar(Ridge(alpha=10.0))
model_svr = TransformedTargetRegressor(
    regressor=pipeline_dasar(SVR(kernel="rbf", C=10.0, epsilon=0.1)),
    transformer=StandardScaler(),
)
metrik = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}
hasil_cv = []
for nama_model, estimator in (("Ridge linear", model_linear), ("SVR RBF nonlinear", model_svr)):
    skor = cross_validate(
        estimator,
        X,
        y,
        cv=cv,
        scoring=metrik,
        error_score="raise",
        n_jobs=1,
    )
    hasil_cv.append(
        {
            "Model": nama_model,
            "MAE rata-rata": -skor["test_MAE"].mean(),
            "MAE std": skor["test_MAE"].std(),
            "RMSE rata-rata": -skor["test_RMSE"].mean(),
            "R2 rata-rata": skor["test_R2"].mean(),
        }
    )

tabel_cv = pd.DataFrame(hasil_cv).sort_values("RMSE rata-rata")
display(tabel_cv.round(3))
print("Dengan MAE/RMSE, nilai lebih kecil lebih baik; R2 lebih besar lebih baik.")

## Panduan interpretasi

- Scatter satu fitur yang mendekati garis dan residual tanpa pola mendukung hubungan linear **untuk fitur tersebut**, bukan semua fitur sekaligus.
- Korelasi besar hanya menunjukkan asosiasi linear univariat; fitur yang terlihat terbaik dipilih dari data yang sama sehingga skornya optimistis.
- Jika Ridge memiliki skor validasi waktu yang sebanding atau lebih baik daripada SVR, model linear teratur adalah baseline yang masuk akal. Jika SVR lebih baik secara konsisten, ada alasan untuk menguji model nonlinear lebih lanjut.
- Nilai R2 negatif pada fold waktu berarti model lebih buruk daripada baseline prediksi rata-rata target di fold tersebut; itu dapat terjadi saat pola berubah sepanjang waktu.
- Jangan standardisasi atau memilih fitur di luar pipeline sebelum cross-validation. Untuk evaluasi ilmiah yang kuat, validasi berdasarkan kultivasi/kelompok independen bila ID kultivasi tersedia.
- Pastikan ulang alignment timestamp gambar dan target jumlah sel. Jika pasangan salah, analisis linearitas dan pemilihan model tidak dapat dipercaya.